# 🧠 Deepfake Detector — Kaggle Edition (Video + Audio)

Two clean, independent detectors, each trained and evaluated end-to-end:

| Branch | Input | Backbone | Output |
|--------|-------|----------|--------|
| **Visual** | face / frame images | EfficientNet-B0 (timm) | P(fake) |
| **Audio** | voice clips → log-mel | compact CNN | P(fake) |

For each branch you get the full metric suite on **train / val / test**:
accuracy · balanced-accuracy · precision · recall(sensitivity) · specificity · F1 ·
ROC-AUC · PR-AUC(AP) · MCC · **EER** · confusion matrix · ROC/PR/score plots.

### How to run
1. **Settings → Accelerator → GPU** (T4/P100).
2. **Add Data** (right panel) — add any of:
   - Visual: `chuneeb/deepfake-detection-dataset-2026`, `ucimachinelearning/deep-fake-detection-cropped-dataset`
   - Audio: `unidpro/real-vs-fake-human-voice-deepfake-audio`, or a Fake-or-Real (FoR) dataset
3. (Optional) **Settings → Internet → ON** if you want the Hugging Face download cell.
4. Run all. The notebook **auto-detects** whatever you mounted and skips the rest.

> Nothing is hard-coded to one dataset path. Add image folders, audio folders, or both.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, subprocess, importlib

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        pip_name or pkg], check=False)

# timm + librosa are usually preinstalled on Kaggle; guard anyway.
_ensure("timm")
_ensure("librosa")
_ensure("soundfile")

import os, glob, random, json, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu":
    print("⚠️  No GPU detected — turn on the GPU accelerator for reasonable speed.")

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
class CFG:
    input_root      = "/kaggle/input"
    out_dir         = "/kaggle/working"

    # Caps so a session finishes comfortably. Raise once you know it runs.
    max_per_class   = 6000      # images per class (real / fake)
    max_audio_per_class = 2500  # audio clips per class

    # Splits (used only when a dataset has no built-in train/val/test folders)
    val_frac, test_frac = 0.15, 0.15

    # Visual
    img_size        = 224
    img_backbone    = "efficientnet_b0"
    img_epochs      = 4
    img_batch       = 64
    img_lr          = 3e-4

    # Audio
    sr              = 16000
    audio_seconds   = 3.0
    n_mels          = 128
    audio_epochs    = 8
    audio_batch     = 64
    audio_lr        = 1e-3

    num_workers     = 2

os.makedirs(CFG.out_dir, exist_ok=True)

# Keyword heuristics for label inference from file paths.
FAKE_KW = ("fake", "deepfake", "df", "spoof", "synthetic", "synth", "ai", "generated",
           "gen", "manipulat", "swap", "clone", "tts", "vc")
REAL_KW = ("real", "genuine", "authentic", "bonafide", "bona-fide", "live",
           "original", "orig", "human", "pristine", "true")

def label_from_path(p):
    # Return 1 (fake), 0 (real), or None if undecidable, from path tokens.
    low = p.lower()
    parts = low.replace("\\", "/").split("/")
    # prefer the most specific (deepest) folder token that matches
    for token in reversed(parts):
        # exact-ish folder names first
        if token in ("fake", "fakes", "1_fake", "spoof", "df", "deepfake"):  return 1
        if token in ("real", "reals", "0_real", "bonafide", "genuine", "live"): return 0
    if any(k in low for k in FAKE_KW) and not any(k in low for k in ("realtime",)):
        # ensure 'real' isn't ALSO present more specifically
        if any(k in low for k in REAL_KW):
            # decide by which keyword appears deeper in the path
            fi = max((low.rfind(k) for k in FAKE_KW if k in low), default=-1)
            ri = max((low.rfind(k) for k in REAL_KW if k in low), default=-1)
            return 1 if fi > ri else 0
        return 1
    if any(k in low for k in REAL_KW):
        return 0
    return None

def split_kind(p):
    # Detect built-in split from path: 'train' / 'val' / 'test' or None.
    low = "/" + p.lower().replace("\\", "/") + "/"
    if "/test/" in low or "/testing/" in low or "/eval/" in low: return "test"
    if "/val/"  in low or "/valid/" in low or "/validation/" in low: return "val"
    if "/train/" in low or "/training/" in low: return "train"
    return None

print("Config ready.")

In [ ]:
# ── Discover mounted datasets ───────────────────────────────────────────────
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
AUD_EXT = (".wav", ".flac", ".mp3", ".ogg", ".m4a")

def scan(root, exts, cap_total=400000):
    found = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(exts):
                found.append(os.path.join(dirpath, f))
                if len(found) >= cap_total:
                    return found
    return found

print("Scanning", CFG.input_root, "…")
all_imgs = scan(CFG.input_root, IMG_EXT)
all_auds = scan(CFG.input_root, AUD_EXT)
print(f"  images found: {len(all_imgs):,}")
print(f"  audio  found: {len(all_auds):,}")

if not all_imgs and not all_auds:
    print("\n⚠️  No data mounted. Use the 'Add Data' panel to attach a dataset, then re-run.")

VIDEO_ON = len(all_imgs) > 50
AUDIO_ON = len(all_auds) > 50
print(f"\nVIDEO branch: {'ON' if VIDEO_ON else 'off'}   |   AUDIO branch: {'ON' if AUDIO_ON else 'off'}")

In [ ]:
# ── OPTIONAL: pull a dataset from Hugging Face (needs Internet = ON) ─────────
# Leave RUN_HF = False unless you want this. Example pulls an image deepfake set.
RUN_HF = False
HF_REPO = "Hemg/deepfake-and-real-images"   # change to any HF image/audio repo

if RUN_HF:
    try:
        from huggingface_hub import snapshot_download
        path = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                                 local_dir="/kaggle/working/hf_data")
        print("Downloaded to", path)
        # rescan so the new files get picked up
        all_imgs = scan("/kaggle/working/hf_data", IMG_EXT) or all_imgs
        all_auds = scan("/kaggle/working/hf_data", AUD_EXT) or all_auds
        VIDEO_ON = len(all_imgs) > 50; AUDIO_ON = len(all_auds) > 50
        print("images:", len(all_imgs), "audio:", len(all_auds))
    except Exception as e:
        print("HF download skipped/failed:", e)
else:
    print("HF download disabled (RUN_HF=False).")

In [ ]:
# ── Build labeled (path,label,split) tables, balanced + capped ───────────────
def build_table(paths, cap_per_class):
    rows = []
    for p in paths:
        y = label_from_path(p)
        if y is None:
            continue
        rows.append((p, y, split_kind(p)))
    if not rows:
        return pd.DataFrame(columns=["path", "label", "split"])
    df = pd.DataFrame(rows, columns=["path", "label", "split"])

    # balance + cap per class
    parts = []
    for lab, g in df.groupby("label"):
        g = g.sample(frac=1.0, random_state=SEED)
        parts.append(g.head(cap_per_class))
    df = pd.concat(parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    # assign splits: honor built-in split if a meaningful one exists, else stratified
    has_builtin = df["split"].notna().mean() > 0.6 and df["split"].nunique() >= 2
    if not has_builtin:
        df["split"] = None
        for lab, g in df.groupby("label"):
            idx = g.index.tolist()
            n = len(idx); n_test = int(n*CFG.test_frac); n_val = int(n*CFG.val_frac)
            df.loc[idx[:n_test], "split"] = "test"
            df.loc[idx[n_test:n_test+n_val], "split"] = "val"
            df.loc[idx[n_test+n_val:], "split"] = "train"
    else:
        # if val missing, carve val out of train
        if (df["split"] == "val").sum() == 0:
            tr = df[df["split"] == "train"]
            carve = tr.sample(frac=CFG.val_frac, random_state=SEED).index
            df.loc[carve, "split"] = "val"
    return df

img_df = build_table(all_imgs, CFG.max_per_class) if VIDEO_ON else pd.DataFrame()
aud_df = build_table(all_auds, CFG.max_audio_per_class) if AUDIO_ON else pd.DataFrame()

def summarize(df, name):
    if len(df) == 0:
        print(f"{name}: (none)"); return
    print(f"\n{name}: {len(df):,} samples")
    print(pd.crosstab(df["split"], df["label"].map({0:"real",1:"fake"}),
                      margins=True))

summarize(img_df, "VISUAL")
summarize(aud_df, "AUDIO")

# guard against label-inference failure (only one class present)
if VIDEO_ON and img_df["label"].nunique() < 2:
    print("\n⚠️  Visual: couldn't infer both classes from paths — check folder names "
          "(expect 'real'/'fake' style). Disabling visual branch.")
    VIDEO_ON = False
if AUDIO_ON and aud_df["label"].nunique() < 2:
    print("\n⚠️  Audio: couldn't infer both classes from paths. Disabling audio branch.")
    AUDIO_ON = False

In [ ]:
# ── Shared metrics + plotting ────────────────────────────────────────────────
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             matthews_corrcoef, balanced_accuracy_score)
import matplotlib.pyplot as plt

def equal_error_rate(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2), float(thr[i])

def best_f1_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1 = 2*prec*rec / (prec + rec + 1e-12)
    if len(thr) == 0:
        return 0.5
    return float(thr[np.nanargmax(f1[:-1])])

def compute_metrics(y_true, y_prob, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    thr = best_f1_threshold(y_true, y_prob) if threshold is None else threshold
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    eps = 1e-12
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)          # sensitivity / TPR (fake = positive)
    specificity = tn / (tn + fp + eps)        # TNR
    f1 = 2*precision*recall / (precision + recall + eps)
    acc = (tp + tn) / (tp + tn + fp + fn + eps)
    try:    auc = roc_auc_score(y_true, y_prob)
    except Exception: auc = float("nan")
    try:    ap  = average_precision_score(y_true, y_prob)
    except Exception: ap = float("nan")
    eer, eer_thr = equal_error_rate(y_true, y_prob)
    return {
        "accuracy": acc, "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision": precision, "recall": recall, "specificity": specificity,
        "f1": f1, "roc_auc": auc, "pr_auc": ap,
        "mcc": matthews_corrcoef(y_true, y_pred) if len(set(y_pred))>1 else 0.0,
        "eer": eer, "threshold": thr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "n": int(len(y_true)),
    }

def print_metrics(name, m):
    print(f"\n── {name} ──  (n={m['n']}, thr={m['threshold']:.3f})")
    order = ["accuracy","balanced_acc","precision","recall","specificity","f1",
             "roc_auc","pr_auc","mcc","eer"]
    for k in order:
        print(f"   {k:<14}: {m[k]:.4f}")
    print(f"   confusion      : TP={m['TP']} FP={m['FP']} TN={m['TN']} FN={m['FN']}")

def plot_eval(name, y_true, y_prob, m, save_prefix):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax[0].plot(fpr, tpr, lw=2, label=f"AUC={m['roc_auc']:.3f}")
    ax[0].plot([0,1],[0,1],"--",c="gray"); ax[0].set_title(f"{name} — ROC")
    ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend(loc="lower right")
    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    ax[1].plot(rec, prec, lw=2, label=f"AP={m['pr_auc']:.3f}")
    ax[1].set_title(f"{name} — Precision-Recall")
    ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend(loc="lower left")
    # confusion matrix
    cm = np.array([[m['TN'], m['FP']],[m['FN'], m['TP']]])
    ax[2].imshow(cm, cmap="Blues")
    ax[2].set_xticks([0,1]); ax[2].set_yticks([0,1])
    ax[2].set_xticklabels(["pred real","pred fake"])
    ax[2].set_yticklabels(["real","fake"])
    for (i,j),v in np.ndenumerate(cm):
        ax[2].text(j, i, str(v), ha="center", va="center",
                   color="white" if v > cm.max()/2 else "black", fontsize=13)
    ax[2].set_title(f"{name} — Confusion (thr={m['threshold']:.2f})")
    plt.tight_layout()
    out = os.path.join(CFG.out_dir, f"{save_prefix}_eval.png")
    plt.savefig(out, dpi=110, bbox_inches="tight"); plt.show()
    print("saved", out)

print("Metrics utilities ready.")

In [ ]:
# ── VISUAL: dataset + EfficientNet-B0 ───────────────────────────────────────
import timm
from PIL import Image
import torchvision.transforms as T

train_tf = T.Compose([
    T.Resize((CFG.img_size, CFG.img_size)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.1, 0.1, 0.1),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
eval_tf = T.Compose([
    T.Resize((CFG.img_size, CFG.img_size)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

class ImgDS(Dataset):
    def __init__(self, df, tf):
        self.p = df["path"].tolist(); self.y = df["label"].tolist(); self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try:
            im = Image.open(self.p[i]).convert("RGB")
        except Exception:
            im = Image.new("RGB", (CFG.img_size, CFG.img_size))
        return self.tf(im), torch.tensor(self.y[i], dtype=torch.float32)

def make_loader(df, tf, bs, shuffle):
    return DataLoader(ImgDS(df, tf), batch_size=bs, shuffle=shuffle,
                      num_workers=CFG.num_workers, pin_memory=True, drop_last=False)

class ImgNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.net = timm.create_model(backbone, pretrained=True, num_classes=1)
    def forward(self, x): return self.net(x).squeeze(1)

print("Visual components ready." if VIDEO_ON else "Visual branch off — skipping.")

In [ ]:
# ── VISUAL: train ───────────────────────────────────────────────────────────
@torch.no_grad()
def infer_probs(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
            p = torch.sigmoid(model(x))
        P.append(p.float().cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(Y), np.concatenate(P)

img_model = None
if VIDEO_ON:
    tr = img_df[img_df.split=="train"]; va = img_df[img_df.split=="val"]
    tl = make_loader(tr, train_tf, CFG.img_batch, True)
    vl = make_loader(va, eval_tf,  CFG.img_batch, False)

    img_model = ImgNet(CFG.img_backbone).to(DEVICE)
    opt = torch.optim.AdamW(img_model.parameters(), lr=CFG.img_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.img_epochs*max(1,len(tl)))
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
    lossfn = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None
    for ep in range(CFG.img_epochs):
        img_model.train(); t0 = time.time(); run = 0
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                loss = lossfn(img_model.net(x).squeeze(1), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()*len(x)
        yv, pv = infer_probs(img_model, vl)
        try: auc = roc_auc_score(yv, pv)
        except Exception: auc = float("nan")
        print(f"  epoch {ep+1}/{CFG.img_epochs}  loss={run/len(tr):.4f}  "
              f"val_auc={auc:.4f}  ({time.time()-t0:.0f}s)")
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in img_model.state_dict().items()}
    if best_state: img_model.load_state_dict(best_state)
    torch.save(img_model.state_dict(), os.path.join(CFG.out_dir, "visual_model.pt"))
    print(f"Visual training done. best val AUC={best_auc:.4f}")
else:
    print("Visual branch off — skipping training.")

In [ ]:
# ── VISUAL: evaluate on train / val / test ──────────────────────────────────
visual_results = {}
if VIDEO_ON and img_model is not None:
    # lock threshold on val, apply to test (no test-set tuning)
    yv, pv = infer_probs(img_model, make_loader(img_df[img_df.split=="val"], eval_tf, CFG.img_batch, False))
    locked_thr = best_f1_threshold(yv, pv)
    for sp in ["train", "val", "test"]:
        d = img_df[img_df.split==sp]
        if len(d)==0: continue
        yy, pp = infer_probs(img_model, make_loader(d, eval_tf, CFG.img_batch, False))
        thr = None if sp=="val" else locked_thr   # val finds its own; test uses locked
        m = compute_metrics(yy, pp, threshold=thr)
        visual_results[sp] = m
        print_metrics(f"VISUAL [{sp}]", m)
        if sp=="test":
            plot_eval("Visual (test)", yy, pp, m, "visual")
else:
    print("Visual branch off — no metrics.")

In [ ]:
# ── AUDIO: log-mel dataset + compact CNN ────────────────────────────────────
import librosa

TARGET_LEN = int(CFG.sr * CFG.audio_seconds)

def load_logmel(path):
    try:
        y, _ = librosa.load(path, sr=CFG.sr, mono=True)
    except Exception:
        y = np.zeros(TARGET_LEN, dtype=np.float32)
    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)))
    else:
        y = y[:TARGET_LEN]
    mel = librosa.feature.melspectrogram(y=y, sr=CFG.sr, n_mels=CFG.n_mels,
                                         n_fft=1024, hop_length=256)
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)            # [n_mels, T]

class AudDS(Dataset):
    def __init__(self, df):
        self.p = df["path"].tolist(); self.y = df["label"].tolist()
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        mel = load_logmel(self.p[i])
        return torch.from_numpy(mel).unsqueeze(0), torch.tensor(self.y[i], dtype=torch.float32)

def aud_loader(df, bs, shuffle):
    return DataLoader(AudDS(df), batch_size=bs, shuffle=shuffle,
                      num_workers=CFG.num_workers, pin_memory=True)

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(ci, co):
            return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co),
                                 nn.ReLU(), nn.MaxPool2d(2))
        self.feat = nn.Sequential(block(1,16), block(16,32), block(32,64), block(64,128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x): return self.head(self.feat(x)).squeeze(1)

print("Audio components ready." if AUDIO_ON else "Audio branch off — skipping.")

In [ ]:
# ── AUDIO: train ────────────────────────────────────────────────────────────
@torch.no_grad()
def infer_probs_aud(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
            p = torch.sigmoid(model(x))
        P.append(p.float().cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(Y), np.concatenate(P)

aud_model = None
if AUDIO_ON:
    tr = aud_df[aud_df.split=="train"]; va = aud_df[aud_df.split=="val"]
    tl = aud_loader(tr, CFG.audio_batch, True)
    vl = aud_loader(va, CFG.audio_batch, False)

    aud_model = AudioCNN().to(DEVICE)
    opt = torch.optim.AdamW(aud_model.parameters(), lr=CFG.audio_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.audio_epochs*max(1,len(tl)))
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
    lossfn = nn.BCEWithLogitsLoss()

    best_auc, best_state = -1, None
    for ep in range(CFG.audio_epochs):
        aud_model.train(); t0 = time.time(); run = 0
        for x, y in tl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                loss = lossfn(aud_model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()*len(x)
        yv, pv = infer_probs_aud(aud_model, vl)
        try: auc = roc_auc_score(yv, pv)
        except Exception: auc = float("nan")
        print(f"  epoch {ep+1}/{CFG.audio_epochs}  loss={run/len(tr):.4f}  "
              f"val_auc={auc:.4f}  ({time.time()-t0:.0f}s)")
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in aud_model.state_dict().items()}
    if best_state: aud_model.load_state_dict(best_state)
    torch.save(aud_model.state_dict(), os.path.join(CFG.out_dir, "audio_model.pt"))
    print(f"Audio training done. best val AUC={best_auc:.4f}")
else:
    print("Audio branch off — skipping training.")

In [ ]:
# ── AUDIO: evaluate on train / val / test ───────────────────────────────────
audio_results = {}
if AUDIO_ON and aud_model is not None:
    yv, pv = infer_probs_aud(aud_model, aud_loader(aud_df[aud_df.split=="val"], CFG.audio_batch, False))
    locked_thr = best_f1_threshold(yv, pv)
    for sp in ["train", "val", "test"]:
        d = aud_df[aud_df.split==sp]
        if len(d)==0: continue
        yy, pp = infer_probs_aud(aud_model, aud_loader(d, CFG.audio_batch, False))
        thr = None if sp=="val" else locked_thr
        m = compute_metrics(yy, pp, threshold=thr)
        audio_results[sp] = m
        print_metrics(f"AUDIO [{sp}]", m)
        if sp=="test":
            plot_eval("Audio (test)", yy, pp, m, "audio")
else:
    print("Audio branch off — no metrics.")

In [ ]:
# ── Final summary table + save ──────────────────────────────────────────────
rows = []
for branch, res in [("visual", visual_results if VIDEO_ON else {}),
                    ("audio",  audio_results  if AUDIO_ON else {})]:
    for sp, m in res.items():
        rows.append({"branch": branch, "split": sp,
                     **{k: round(m[k], 4) for k in
                        ["accuracy","precision","recall","specificity","f1",
                         "roc_auc","pr_auc","mcc","eer"]},
                     "n": m["n"]})

summary = pd.DataFrame(rows)
if len(summary):
    cols = ["branch","split","accuracy","precision","recall","specificity","f1",
            "roc_auc","pr_auc","mcc","eer","n"]
    summary = summary[cols]
    print("\n================ FINAL METRICS ================\n")
    print(summary.to_string(index=False))
    summary.to_csv(os.path.join(CFG.out_dir, "metrics_summary.csv"), index=False)
    with open(os.path.join(CFG.out_dir, "metrics_full.json"), "w") as f:
        json.dump({"visual": visual_results, "audio": audio_results}, f, indent=2)
    print("\nSaved → metrics_summary.csv, metrics_full.json, *_eval.png, *_model.pt")
else:
    print("No results — add a dataset (Add Data panel) and re-run.")
summary